**1. Importing the dependencies**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder,StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,precision_score,recall_score,roc_auc_score,precision_recall_curve
import pickle

**2. Data Loading and Understanding**

In [2]:
# load teh csv data to a pandas dataframe
df = pd.read_csv("../data/cleaned.csv")

In [3]:
df.shape

(7043, 20)

In [4]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


Label encoding of categorical fetaures

In [5]:
df_encoded=df.copy()

In [6]:
# identifying columns with object data type
object_columns = df_encoded.select_dtypes(include="str").columns

In [7]:
print(object_columns)

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='str')


In [8]:
# initialize a dictionary to save the encoders
encoders = {}

# apply label encoding and store the encoders
for column in object_columns:
  label_encoder = LabelEncoder()
  df_encoded[column] = label_encoder.fit_transform(df[column])
  encoders[column] = label_encoder


In [9]:
encoders

{'gender': LabelEncoder(),
 'Partner': LabelEncoder(),
 'Dependents': LabelEncoder(),
 'PhoneService': LabelEncoder(),
 'MultipleLines': LabelEncoder(),
 'InternetService': LabelEncoder(),
 'OnlineSecurity': LabelEncoder(),
 'OnlineBackup': LabelEncoder(),
 'DeviceProtection': LabelEncoder(),
 'TechSupport': LabelEncoder(),
 'StreamingTV': LabelEncoder(),
 'StreamingMovies': LabelEncoder(),
 'Contract': LabelEncoder(),
 'PaperlessBilling': LabelEncoder(),
 'PaymentMethod': LabelEncoder()}

In [10]:
df_encoded.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15,1
3,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,1840.75,0
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,151.65,1


**Training and test data split**

In [11]:
# splitting the features and target
X = df_encoded.drop(columns=["Churn"])
y = df_encoded["Churn"]

In [12]:
# split training and test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
ss=StandardScaler()
X_train_scaled=ss.fit_transform(X_train)
X_test_scaled=ss.transform(X_test)

In [14]:
print(y_train.shape)

(5634,)


In [15]:
print(y_train.value_counts())

Churn
0    4138
1    1496
Name: count, dtype: int64


**Applying Synthetic Minority Oversampling Technique (SMOTE)**

In [16]:
smote = SMOTE(random_state=42)

In [17]:
X_train_scaled_smote,y_train_smote1=smote.fit_resample(X_train_scaled,y_train)

In [18]:
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [19]:
print(y_train_smote.shape)

(8276,)


In [20]:
print(y_train_smote.value_counts())

Churn
0    4138
1    4138
Name: count, dtype: int64


**5. Model Training**

Training with default hyperparameters

In [21]:
lr=LogisticRegression(random_state=42)

In [22]:
lr.fit(X_train_scaled_smote,y_train_smote1)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solve

In [23]:
y_prob = lr.predict_proba(X_test_scaled)[:, 1]
import numpy as np
from sklearn.metrics import f1_score

thresholds = np.arange(0.20, 0.81, 0.01)

best_threshold = 0
best_f1 = 0

for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    f1 = f1_score(y_test, y_pred)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("Best Threshold:", round(best_threshold,2))
print("Best F1:", round(best_f1,4))
y_pred = (y_prob >= best_threshold).astype(int)

Best Threshold: 0.68
Best F1: 0.6484


In [95]:
print("Accuracy :", accuracy_score(y_test,y_pred))
print("Precision:", precision_score(y_test,y_pred))
print("Recall   :", recall_score(y_test,y_pred))
print("F1 Score :", f1_score(y_test,y_pred))
print("ROC AUC  :", roc_auc_score(y_test,y_prob))

print("\nClassification Report\n")
print(classification_report(y_test,y_pred))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test,y_pred))

Accuracy : 0.8090844570617459
Precision: 0.6326530612244898
Recall   : 0.6648793565683646
F1 Score : 0.6483660130718955
ROC AUC  : 0.8603400892274886

Classification Report

              precision    recall  f1-score   support

           0       0.88      0.86      0.87      1036
           1       0.63      0.66      0.65       373

    accuracy                           0.81      1409
   macro avg       0.75      0.76      0.76      1409
weighted avg       0.81      0.81      0.81      1409


Confusion Matrix

[[892 144]
 [125 248]]


In [69]:
# dictionary of models
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42),
}

In [70]:
model_list = []
auc_list = []
accuracy_list = []
precision_list = []
recall_list = []
f1_list = []

for name, model in models.items():

  # Train model
  model.fit(X_train_smote, y_train_smote)

  # Cross-validation score
  cv_auc = cross_val_score(
      model,
      X_train,
      y_train,
      cv=5,
      scoring='roc_auc',
      n_jobs=-1
  )

  # Prediction probabilities
  y_prob = model.predict_proba(X_test)[:, 1]

  # Optimal threshold
  precisions, recalls, thresholds = precision_recall_curve(
      y_test,
      y_prob
  )

  f1_scores = (
      2 * precisions * recalls /
      (precisions + recalls + 1e-9)
  )

  best_thresh = thresholds[np.argmax(f1_scores)]

  # Final predictions
  y_pred = (y_prob >= best_thresh).astype(int)

  # ROC-AUC
  test_auc = roc_auc_score(y_test, y_prob)

  # Store results
  model_list.append(name)
  auc_list.append(test_auc)
  accuracy_list.append(accuracy_score(y_test, y_pred))
  precision_list.append(precision_score(y_test, y_pred))
  recall_list.append(recall_score(y_test, y_pred))
  f1_list.append(f1_score(y_test, y_pred))

  # Print Results
  print(name)

  print("Cross Validation Performance")
  print("Mean ROC-AUC: {:.4f}".format(cv_auc.mean()))
  print("Std ROC-AUC : {:.4f}".format(cv_auc.std()))

  print("-" * 40)

  print("Test Set Performance")
  print("Optimal Threshold : {:.3f}".format(best_thresh))
  print("ROC-AUC Score     : {:.4f}".format(test_auc))

  print("\nClassification Report")
  print(classification_report(
      y_test,
      y_pred,
      target_names=['Retained', 'Churned']
  ))

  print("=" * 60)
  print("\n")

Decision Tree
Cross Validation Performance
Mean ROC-AUC: 0.6620
Std ROC-AUC : 0.0158
----------------------------------------
Test Set Performance
Optimal Threshold : 1.000
ROC-AUC Score     : 0.6717

Classification Report
              precision    recall  f1-score   support

    Retained       0.83      0.80      0.81      1036
     Churned       0.49      0.55      0.52       373

    accuracy                           0.73      1409
   macro avg       0.66      0.67      0.67      1409
weighted avg       0.74      0.73      0.74      1409



Random Forest
Cross Validation Performance
Mean ROC-AUC: 0.8233
Std ROC-AUC : 0.0119
----------------------------------------
Test Set Performance
Optimal Threshold : 0.280
ROC-AUC Score     : 0.8233

Classification Report
              precision    recall  f1-score   support

    Retained       0.93      0.67      0.78      1036
     Churned       0.48      0.86      0.62       373

    accuracy                           0.72      1409
   macr

In [71]:
results_df = pd.DataFrame({
    "Model": model_list,
    "Accuracy":accuracy_list,
    "ROC-AUC": auc_list,
    "Precision":precision_list,
    "Recall":recall_list,
    "f1 score":f1_list
})

results_df = results_df.sort_values(by="ROC-AUC", ascending=False).reset_index(drop=True)

results_df

,Model,Accuracy,ROC-AUC,Precision,Recall,f1 score
0,XGBoost,0.758694,0.830043,0.529837,0.785523,0.632829
1,Random Forest,0.718950,0.823278,0.482655,0.857909,0.617761
2,Decision Tree,0.731015,0.671670,0.492754,0.546917,0.518424


Logistic Regression gives the highest accuracy compared to other models with default parameters

### Hyperparameter tuning of Logistic Regression

In [86]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear'],
    'class_weight': [None, 'balanced'],
    'max_iter': [1000]
}

lr2 = LogisticRegression(random_state=42)

grid = GridSearchCV(
    estimator=lr2,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV ROC-AUC:", grid.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


c:\Users\Lenovo\Downloads\Anaconda3\envs\v_env\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Best Parameters: {'C': 1, 'class_weight': None, 'max_iter': 1000, 'penalty': 'l2', 'solver': 'lbfgs'}
Best CV ROC-AUC: 0.83973151962867


c:\Users\Lenovo\Downloads\Anaconda3\envs\v_env\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [87]:
best_lr = grid.best_estimator_
best_lr

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'l2'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To ch

In [88]:
y_prob = best_lr.predict_proba(X_test_scaled)[:, 1]

c:\Users\Lenovo\Downloads\Anaconda3\envs\v_env\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [89]:
thresholds = np.arange(0.20, 0.81, 0.01)

best_threshold = 0
best_f1 = 0

for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    f1 = f1_score(y_test, y_pred)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("Best Threshold:", round(best_threshold,2))
print("Best F1:", round(best_f1,4))

Best Threshold: 0.4
Best F1: 0.5983


In [90]:
y_pred = (y_prob >= best_threshold).astype(int)

In [91]:

print("Accuracy :", accuracy_score(y_test,y_pred))
print("Precision:", precision_score(y_test,y_pred))
print("Recall   :", recall_score(y_test,y_pred))
print("F1 Score :", f1_score(y_test,y_pred))
print("ROC AUC  :", roc_auc_score(y_test,y_prob))

print("\nClassification Report\n")
print(classification_report(y_test,y_pred))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test,y_pred))

Accuracy : 0.6941092973740242
Precision: 0.4585714285714286
Recall   : 0.8605898123324397
F1 Score : 0.5983224603914259
ROC AUC  : 0.7887536617429378

Classification Report

              precision    recall  f1-score   support

           0       0.93      0.63      0.75      1036
           1       0.46      0.86      0.60       373

    accuracy                           0.69      1409
   macro avg       0.69      0.75      0.68      1409
weighted avg       0.80      0.69      0.71      1409


Confusion Matrix

[[657 379]
 [ 52 321]]


### Hyperparameter Tuning of Random Forest

In [78]:
rf = RandomForestClassifier(random_state=42)

In [79]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'class_weight': [None, 'balanced']
}

In [80]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=30,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

In [81]:
random_search.fit(X_train_smote, y_train_smote)

best_rf = random_search.best_estimator_

print("Best Parameters:")
print(random_search.best_params_)

print("\nBest CV ROC-AUC:")
print(random_search.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters:
{'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None, 'class_weight': 'balanced'}

Best CV ROC-AUC:
0.9248518099384067


In [82]:
y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:,1]

In [83]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print(f"Accuracy : {accuracy_score(y_test,y_pred):.4f}")
print(f"Precision: {precision_score(y_test,y_pred):.4f}")
print(f"Recall   : {recall_score(y_test,y_pred):.4f}")
print(f"F1 Score : {f1_score(y_test,y_pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test,y_prob):.4f}")

print("\nClassification Report")
print(classification_report(y_test,y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test,y_pred))

Accuracy : 0.7729
Precision: 0.5722
Recall   : 0.5630
F1 Score : 0.5676
ROC AUC  : 0.8257

Classification Report
              precision    recall  f1-score   support

           0       0.84      0.85      0.85      1036
           1       0.57      0.56      0.57       373

    accuracy                           0.77      1409
   macro avg       0.71      0.71      0.71      1409
weighted avg       0.77      0.77      0.77      1409


Confusion Matrix
[[879 157]
 [163 210]]


Selecting deafult tuned Logistic Regression model as it achieved the highest ROC-AUC (0.86), F1-score (0.65), precision, recall, and overall accuracy.

In [123]:
X_scaled=ss.transform(X)
churn_probabilities = lr.predict_proba(X_scaled)[:, 1]

# Add probabilities to the dataframe
df['Churn_Probability'] = churn_probabilities

# View first few rows
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Churn_Probability
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,0.830306
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,No,No,One year,No,Mailed check,56.95,1889.50,0,0.107333
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,0.552782
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0,0.062844
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,0.849347


In [124]:
df.to_csv('churn_prob.csv',index=False)

**7. Save everything in one bundle**

In [132]:
import os 
os.makedirs("../model", exist_ok=True)

bundle = {
    "model": lr,
    "scaler": ss,
    "encoders": encoders,
    "feature_names": X.columns.tolist(),
}

with open(os.path.join("../model", "model_bundle.pkl"), "wb") as f:
    pickle.dump(bundle, f)